# 1. Data profiling

Inspect the original source before deciding which records belong in the sales dataset.

Run `python -m pip install -e ".[dev]"` from the repository root first. Each notebook runs independently and imports the same tested Python functions. Source timestamps retain their original local calendar meaning. Set `RETAIL_ETL_INPUT` to use another source and adjust `AS_OF` to its reporting cutoff.

In [ ]:
import os
from pathlib import Path
import pandas as pd
from retail_etl.pipeline import project_root, read_source
from retail_etl.transform import prepare_data, feature_engineering, validate_output

ROOT = project_root()
SOURCE = Path(os.getenv("RETAIL_ETL_INPUT", str(ROOT / "data/raw/Online Retail.xlsx")))
OUTPUT = Path(os.getenv("RETAIL_ETL_OUTPUT", str(ROOT / "outputs")))
AS_OF = "2011-12-09"  # Inclusive end date of the bundled historical dataset.


## Read the original source

Keep the source unchanged. One row is an invoice line, not an entire order.

In [ ]:
raw = read_source(SOURCE)
print(f"Raw rows: {len(raw):,}; source columns: {len(raw.columns)}")
raw.head()

## Completeness and observed types

In [ ]:
profile = pd.DataFrame({"dtype": raw.dtypes.astype(str), "missing": raw.isna().sum(), "unique": raw.nunique()})
profile

## Exact duplicates

Repeated invoices or products are not automatically duplicates. The project removes exact source rows under an explicit, documented policy.

In [ ]:
print("Exact duplicate rows:", raw.duplicated().sum())
raw.describe(include="all")

## Explore validation rules

Review counts can overlap. Missing customer IDs and returns represent analytical exclusions rather than necessarily incorrect transactions.

In [ ]:
prepared = prepare_data(raw, as_of=AS_OF)
pd.Series(prepared.validation["rule_counts"], name="flagged_rows")

The standalone CLI saves durable validation reports and rejected rows. This notebook only explores data and does not overwrite pipeline exports.